In [10]:
from pathlib import Path
import sys
import re
from collections import Counter,defaultdict

project_root = Path.cwd().resolve()
if str(project_root / "src") not in sys.path:
    sys.path.append(str(project_root / "src"))

from imdb_sentiment.features.preprocess import normalize_review_text
from imdb_sentiment.data.lstm import tokenize_lstm_text,build_lstm_vocabulary
from imdb_sentiment.data.dataset import load_imdb_dataset


In [5]:
dataset = load_imdb_dataset()
train_texts = list(dataset["train"]["text"])

In [2]:
examples = [
    "This movie was amazing!!!",
    "I don't recommend it...",
    "Worst-film-ever",
    "It was <br /><br /> really good",
    "10/10, would watch again.",
    "Meh... not bad, actually.",
    "I loved the cast, but the plot was weak.",
    "Terrible??? No. Just boring.",
    "A must-see!!!",
    "It's okay -- not great, not awful.",
]

In [3]:
for i, text in enumerate(examples, start=1):
    normalized = normalize_review_text(text)
    tokens = tokenize_lstm_text(text)

    print("=" * 80)
    print(f"EXAMPLE {i}")
    print("RAW:       ", text)
    print("NORMALIZED:", normalized)
    print("TOKENS:    ", tokens)
    print("N TOKENS:  ", len(tokens))

EXAMPLE 1
RAW:        This movie was amazing!!!
NORMALIZED: this movie was amazing!!!
TOKENS:     ['this', 'movie', 'was', 'amazing!!!']
N TOKENS:   4
EXAMPLE 2
RAW:        I don't recommend it...
NORMALIZED: i don't recommend it...
TOKENS:     ['i', "don't", 'recommend', 'it...']
N TOKENS:   4
EXAMPLE 3
RAW:        Worst-film-ever
NORMALIZED: worst-film-ever
TOKENS:     ['worst-film-ever']
N TOKENS:   1
EXAMPLE 4
RAW:        It was <br /><br /> really good
NORMALIZED: it was really good
TOKENS:     ['it', 'was', 'really', 'good']
N TOKENS:   4
EXAMPLE 5
RAW:        10/10, would watch again.
NORMALIZED: 10/10, would watch again.
TOKENS:     ['10/10,', 'would', 'watch', 'again.']
N TOKENS:   4
EXAMPLE 6
RAW:        Meh... not bad, actually.
NORMALIZED: meh... not bad, actually.
TOKENS:     ['meh...', 'not', 'bad,', 'actually.']
N TOKENS:   4
EXAMPLE 7
RAW:        I loved the cast, but the plot was weak.
NORMALIZED: i loved the cast, but the plot was weak.
TOKENS:     ['i', 'loved', 'the

In [6]:
counter = Counter()
for text in train_texts:
    counter.update(tokenize_lstm_text(text))

punct_attached = []
hyphenated = []
only_punct = []

for token, count in counter.items():
    if re.search(r"[a-z0-9][^\w\s]+$", token):   # слово с прилипшей пунктуацией в конце
        punct_attached.append((token, count))
    if "-" in token and re.search(r"[a-zA-Z]-[a-zA-Z]", token):  # дефисные слова
        hyphenated.append((token, count))
    if re.fullmatch(r"[^\w\s]+", token):  # токены только из пунктуации
        only_punct.append((token, count))

punct_attached = sorted(punct_attached, key=lambda x: -x[1])[:30]
hyphenated = sorted(hyphenated, key=lambda x: -x[1])[:30]
only_punct = sorted(only_punct, key=lambda x: -x[1])[:30]

print("TOP punct-attached tokens:")
for t, c in punct_attached:
    print(f"{t:20s} {c}")

print("\nTOP hyphenated tokens:")
for t, c in hyphenated:
    print(f"{t:20s} {c}")

print("\nTOP punctuation-only tokens:")
for t, c in only_punct:
    print(f"{t:20s} {c}")

print("\nCounts:")
print("vocab size:", len(counter))
print("punct-attached unique tokens:", len([1 for token in counter if re.search(r'[a-z0-9][^\\w\\s]+$', token)]))
print("hyphenated unique tokens:", len([1 for token in counter if '-' in token and re.search(r'[a-zA-Z]-[a-zA-Z]', token)]))
print("punctuation-only unique tokens:", len([1 for token in counter if re.fullmatch(r'[^\\w\\s]+', token)]))

TOP punct-attached tokens:
it.                  6401
movie.               6086
film.                5202
film,                4043
movie,               3981
it,                  3007
however,             2541
time.                2236
well,                1864
them.                1495
all,                 1473
time,                1381
is,                  1369
one.                 1357
him.                 1328
this.                1320
me.                  1304
that,                1287
story.               1262
well.                1256
all.                 1219
story,               1194
good.                1191
life.                1173
and,                 1145
this,                1140
me,                  1062
movies.              1040
that.                1018
mr.                  1005

TOP hyphenated tokens:
sci-fi               478
low-budget           242
over-the-top         164
so-called            161
film-making          155
make-up              152
b-movie            

In [7]:
VOCAB_SIZE = 30000
SPECIAL_TOKENS = 2 
TOP_K = VOCAB_SIZE - SPECIAL_TOKENS
sorted_tokens = sorted(counter.items(), key=lambda x: (-x[1], x[0]))
top_tokens = sorted_tokens[:TOP_K]

def is_punct_attached(token: str) -> bool:
    return bool(re.search(r"[a-z0-9][^\w\s]+$", token))

def is_hyphenated(token: str) -> bool:
    return "-" in token and bool(re.search(r"[a-zA-Z]-[a-zA-Z]", token))

def is_punctuation_only(token: str) -> bool:
    return bool(re.fullmatch(r"[^\w\s]+", token))

punct_attached_top = [(t, c) for t, c in top_tokens if is_punct_attached(t)]
hyphenated_top = [(t, c) for t, c in top_tokens if is_hyphenated(t)]
punct_only_top = [(t, c) for t, c in top_tokens if is_punctuation_only(t)]

print(f"Top vocab size (without pad/unk): {TOP_K}")
print(f"Punct-attached in top vocab: {len(punct_attached_top)}")
print(f"Hyphenated in top vocab: {len(hyphenated_top)}")
print(f"Punctuation-only in top vocab: {len(punct_only_top)}")

print("\nTop 30 punct-attached tokens in model vocab:")
for t, c in punct_attached_top[:30]:
    print(f"{t:20s} {c}")

print("\nTop 30 punctuation-only tokens in model vocab:")
for t, c in punct_only_top[:30]:
    print(f"{t:20s} {c}")

print("\nTop 30 hyphenated tokens in model vocab:")
for t, c in hyphenated_top[:30]:
    print(f"{t:20s} {c}")

Top vocab size (without pad/unk): 29998
Punct-attached in top vocab: 9703
Hyphenated in top vocab: 514
Punctuation-only in top vocab: 64

Top 30 punct-attached tokens in model vocab:
it.                  6401
movie.               6086
film.                5202
film,                4043
movie,               3981
it,                  3007
however,             2541
time.                2236
well,                1864
them.                1495
all,                 1473
time,                1381
is,                  1369
one.                 1357
him.                 1328
this.                1320
me.                  1304
that,                1287
story.               1262
well.                1256
all.                 1219
story,               1194
good.                1191
life.                1173
and,                 1145
this,                1140
me,                  1062
movies.              1040
that.                1018
mr.                  1005

Top 30 punctuation-only tokens in mo

In [ ]:
sorted_tokens = sorted(counter.items(), key=lambda x: (-x[1], x[0]))
top_tokens = sorted_tokens[:TOP_K]

def strip_edge_punct(token: str) -> str:
    return re.sub(r"^[^\w']+|[^\w']+$", "", token)

families = defaultdict(list)

for token, count in top_tokens:
    base = strip_edge_punct(token)
    if base and base != token:
        families[base].append((token, count))

fragmented_families = []
for base, variants in families.items():
    if len(variants) >= 2:
        total_variant_count = sum(count for _, count in variants)
        fragmented_families.append((base, total_variant_count, len(variants), sorted(variants, key=lambda x: -x[1])))

fragmented_families = sorted(fragmented_families, key=lambda x: -x[1])

print("TOP fragmented token families:\n")
for base, total_count, n_variants, variants in fragmented_families[:30]:
    print(f"BASE: {base:15s} | total_count={total_count:6d} | variants={n_variants}")
    print("   ", variants[:10])
    print()

print("Number of fragmented families in top vocab:", len(fragmented_families))

TOP fragmented token families:

BASE: movie           | total_count= 11569 | variants=24
    [('movie.', 6086), ('movie,', 3981), ('movie!', 342), ('movie?', 236), ('movie:', 165), ('movie...', 103), ('movie)', 102), ('movie;', 85), ('movie"', 82), ('movie).', 72)]

BASE: it              | total_count= 11403 | variants=27
    [('it.', 6401), ('it,', 3007), ('it!', 536), ('it?', 308), ('(it', 186), ('it...', 119), ('it)', 110), ('it:', 96), ('it;', 90), ('it).', 83)]

BASE: film            | total_count= 10321 | variants=21
    [('film.', 5202), ('film,', 4043), ('film!', 172), ('film?', 161), ('film:', 117), ('film;', 114), ('film)', 113), ('film).', 91), ('"film"', 60), ('film...', 59)]

BASE: the             | total_count=  4470 | variants=12
    [('"the', 2791), ('(the', 1393), ('("the', 64), ('`the', 59), ('-the', 39), ('.the', 27), ('the,', 27), (',the', 19), ('*the', 16), ('...the', 13)]

BASE: time            | total_count=  4052 | variants=14
    [('time.', 2236), ('time,', 138

In [13]:
SEED = 42
train_val_split = dataset["train"].train_test_split(test_size=0.2, seed=SEED)

train_texts = list(train_val_split["train"]["text"])
val_texts = list(train_val_split["test"]["text"])
vocab = build_lstm_vocabulary(
    texts=train_texts,
    max_size=VOCAB_SIZE,
)

token_counter = Counter()
unk_counter = Counter()

total_tokens = 0
unk_tokens = 0

for text in val_texts:
    tokens = tokenize_lstm_text(text)
    for token in tokens:
        total_tokens += 1
        token_counter[token] += 1
        if token not in vocab.token_to_id:
            unk_tokens += 1
            unk_counter[token] += 1

unk_rate = unk_tokens / total_tokens if total_tokens > 0 else 0.0

print(f"Validation total tokens: {total_tokens}")
print(f"Validation UNK tokens:   {unk_tokens}")
print(f"Validation UNK rate:     {unk_rate:.4%}")

print("\nTop 50 UNK tokens:")
for token, count in unk_counter.most_common(50):
    print(f"{token:25s} {count}")

Validation total tokens: 1147868
Validation UNK tokens:   77775
Validation UNK rate:     6.7756%

Top 50 UNK tokens:
dominick                  25
borg                      16
nadia                     14
pimlico                   13
yuzna                     13
wray                      13
becky                     13
feat.                     13
gruner                    13
miya                      13
maradona                  13
pinjar                    13
trish                     12
stormtroopers             11
triads                    11
"anna"                    11
petiot                    10
2".                       10
bernie                    10
brody                     10
celie                     10
hopalong                  10
cognac!!!!!i              10
chomsky                   10
rotj                      10
surfing,                  10
kirkland                  10
danni                     9
indistinguishable         9
execs                     9
"boogie         

In [14]:
def strip_edge_punct(token: str) -> str:
    return re.sub(r"^[^\w']+|[^\w']+$", "", token)

unk_counter = Counter()
avoidable_unk_counter = Counter()
total_tokens = 0
unk_tokens = 0
avoidable_unk_tokens = 0

for text in val_texts:
    tokens = tokenize_lstm_text(text)
    for token in tokens:
        total_tokens += 1

        if token not in vocab.token_to_id:
            unk_tokens += 1
            unk_counter[token] += 1

            stripped = strip_edge_punct(token)

            if stripped and stripped != token and stripped in vocab.token_to_id:
                avoidable_unk_tokens += 1
                avoidable_unk_counter[(token, stripped)] += 1

unk_rate = unk_tokens / total_tokens if total_tokens else 0.0
avoidable_share_of_unk = avoidable_unk_tokens / unk_tokens if unk_tokens else 0.0
avoidable_share_of_all = avoidable_unk_tokens / total_tokens if total_tokens else 0.0

print(f"Validation total tokens:           {total_tokens}")
print(f"Validation UNK tokens:             {unk_tokens}")
print(f"Validation UNK rate:               {unk_rate:.4%}")
print()
print(f"Avoidable UNK tokens:              {avoidable_unk_tokens}")
print(f"Share of UNK that is avoidable:    {avoidable_share_of_unk:.4%}")
print(f"Share of all tokens avoidable UNK: {avoidable_share_of_all:.4%}")

print("\nTop 50 avoidable UNK examples:")
for (raw_token, stripped_token), count in avoidable_unk_counter.most_common(50):
    print(f"{raw_token:25s} -> {stripped_token:20s} {count}")

Validation total tokens:           1147868
Validation UNK tokens:             77775
Validation UNK rate:               6.7756%

Avoidable UNK tokens:              30052
Share of UNK that is avoidable:    38.6397%
Share of all tokens avoidable UNK: 2.6181%

Top 50 avoidable UNK examples:
feat.                     -> feat                 13
"anna"                    -> anna                 11
2".                       -> 2                    10
surfing,                  -> surfing              10
"boogie                   -> boogie               9
1993.                     -> 1993                 8
convoluted,               -> convoluted           8
streets"                  -> streets              8
"panic                    -> panic                8
match-                    -> match                8
israel.                   -> israel               7
universal,                -> universal            7
desperate.                -> desperate            7
"miami                    -> mia

## Conclusion of current LSTM preprocessing audit

Current LSTM preprocessing is too coarse for a word-level sentiment model.

Main findings:
- Tokenization is currently based on simple whitespace split after basic normalization.
- Punctuation frequently sticks to words, producing noisy tokens such as `movie.`, `movie,`, `good.`, `terrible???`.
- In the effective model vocabulary (top 30k), 9703 tokens contain punctuation-attached word forms.
- 3180 base word families are fragmented into multiple noisy variants, for example:
  - `movie -> movie., movie,, movie!, ...`
  - `film -> film., film,, film!, ...`
  - `it -> it., it,, it!, ...`
- Validation UNK rate is 6.78%.
- 38.64% of validation UNK tokens are avoidable if edge punctuation is removed.

Engineering conclusion:
The next improvement should target tokenization quality, not model architecture.
The most justified next step is regex-based tokenization that:
1. separates punctuation from words,
2. splits hyphenated compounds,
3. preserves contractions with apostrophes such as `don't` and `it's`.